In [ ]:
import sqlite3
import pandas as pd
import seaborn as sns
import numpy as np
from matplotlib import pyplot as plt
pd.options.display.max_rows = 20
pd.options.display.max_columns = 90

# Loading sql

In [ ]:
import os, shutil

source = "gas_monitoring.db.example"
target = "gas_monitoring.db"

if not os.path.exists(source):
    raise FileNotFoundError(
        f"{source} not found — did you clone the full repo?"
    )
if os.path.exists(target):
    os.remove(target)
shutil.copy2(source, target)
print(f"Copied {source} → {target}")

In [ ]:
con = sqlite3.connect('gas_monitoring.db')

In [ ]:
cursor = con.cursor()

        # Query the sqlite_master table to get table names
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")

        # Fetch all results
table_names = [row[0] for row in cursor.fetchall()]


In [ ]:
query = "SELECT * FROM gas_monitoring"  # Replace 'your_table_name' with the actual table name
df = pd.read_sql_query(query, con)

In [ ]:
df.info()

# Initial Analysis

### Temperature

In [ ]:
display(df['Temperature'].describe())

In [ ]:
anomalies_temp = df[df['Temperature'] > 100]
print(f"Found {len(anomalies_temp)} rows with Temperature > 100")
display(anomalies_temp.head(10))

In [ ]:
df['Temperature'].value_counts(bins=5).sort_index().plot(kind='bar')
plt.title('Temperature Distribution)')
plt.show()

### Humidity

In [ ]:
display(df['Humidity'].describe())
display(df['Humidity'].isna().sum() / df.shape[0])
neg_humidity = df[df['Humidity'] < 0]
print(f"Found {len(neg_humidity)} rows with negative Humidity")
display(neg_humidity.head(10))

In [ ]:
df['Humidity'].value_counts(bins=5).sort_index().plot(kind='bar')
plt.title('Humidity Distribution')
plt.show()

### CO2_InfraredSensor

In [ ]:
display(df[['CO2_InfraredSensor']].describe())

In [ ]:
df['CO2_InfraredSensor'].value_counts(bins=5).sort_index().plot(kind='bar')
plt.title('CO2 Infrared Sensor Distribution')
plt.show()

### CO2_ElectroChemicalSensor

In [ ]:
display(df['CO2_ElectroChemicalSensor'].describe())

In [ ]:
df['CO2_ElectroChemicalSensor'].value_counts(bins=5).sort_index().plot(kind='bar')
plt.title('CO2 ElectroChemical Sensor Distribution')
plt.show()

### CO_GasSensor

In [ ]:
display(df['CO_GasSensor'].value_counts())
display(df['CO_GasSensor'].isna().sum() / df.shape[0])

In [ ]:
df['CO_GasSensor'].value_counts().plot(kind='bar')
plt.title('CO Gas Sensor Distribution')
plt.show()

### Metal Oxide Sensors (Units 1–4)

In [ ]:
mo_cols = [
    'MetalOxideSensor_Unit1',
    'MetalOxideSensor_Unit2',
    'MetalOxideSensor_Unit3',
    'MetalOxideSensor_Unit4'
]
display(df[mo_cols].describe())

In [ ]:
display(df[mo_cols].isna().sum())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for ax, col in zip(axes.ravel(), mo_cols):
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f'{col} Distribution')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df[mo_cols])
plt.title('Metal Oxide Sensor Readings Comparison')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[mo_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Metal Oxide Sensor Correlation')
plt.show()

In [ ]:
sns.pairplot(df[mo_cols])
plt.show()

In [ ]:
sns.pairplot(df[mo_cols + ['Temperature', 'Humidity']])
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
df.groupby('Time of Day')[mo_cols].mean().plot(kind='bar')
plt.title('Average Metal Oxide Readings by Time of Day')
plt.legend(loc='best')
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
df.groupby('HVAC Operation Mode')[mo_cols].mean().plot(kind='bar')
plt.title('Average Metal Oxide Readings by HVAC Operation Mode')
plt.legend(loc='best')
plt.xticks(rotation=45)
plt.show()

### HVAC Operation Mode

In [ ]:
display(df['HVAC Operation Mode'].value_counts())

In [ ]:
df['HVAC Operation Mode'].value_counts().plot(kind='bar')
plt.title('HVAC Operation Mode Distribution')
plt.xticks(rotation=45)
plt.show()

### Ambient Light Level

In [ ]:
display(df['Ambient Light Level'].value_counts())

In [ ]:
df['Ambient Light Level'].value_counts().plot(kind='bar')
plt.title('Ambient Light Level Distribution')
plt.xticks(rotation=45)
plt.show()

### Activity Level

In [ ]:
display(df['Activity Level'].value_counts())

In [ ]:
df['Activity Level'].value_counts().plot(kind='bar')
plt.title('Activity Level Distribution')
plt.xticks(rotation=45)
plt.show()

# Column by Column Data Cleaning

### Temperature

In [ ]:
# Replace unrealistic Temperature values (> 60°C) with NaN
df.loc[df['Temperature'] > 60, 'Temperature'] = np.nan
temp_median = df['Temperature'].median()
df['Temperature'] = df['Temperature'].fillna(temp_median)

In [ ]:
display(df['Temperature'].describe())

In [ ]:
display(df['Temperature'].isna().sum())

### Humidity

In [ ]:
# Replace negative Humidity values with NaN (physically impossible)
df.loc[df['Humidity'] < 0, 'Humidity'] = np.nan
humidity_median = df['Humidity'].median()
df['Humidity'] = df['Humidity'].fillna(humidity_median)

In [ ]:
display(df['Humidity'].describe())

In [ ]:
display(df['Humidity'].isna().sum())

### CO2_InfraredSensor

In [ ]:
#there is no error or missing values for C02 Infrared Sensor
display(df['CO2_InfraredSensor'].isna().sum())

### CO2_ElectroChemicalSensor

In [ ]:
#there is no error or missing values for C02 ElectroChemical Sensor
display(df['CO2_InfraredSensor'].isna().sum())

### CO_GasSensor

In [ ]:
co_median = df['CO_GasSensor'].median()
df['CO_GasSensor'] = df['CO_GasSensor'].fillna(co_median)
df['CO_GasSensor'] = df['CO_GasSensor'].astype(int)

In [ ]:
display(df['CO_GasSensor'].isna().sum())

In [ ]:
display(df['CO_GasSensor'].value_counts())

### Metal Oxide Sensors

In [ ]:
mo_cols = [
    'MetalOxideSensor_Unit1',
    'MetalOxideSensor_Unit2',
    'MetalOxideSensor_Unit3',
    'MetalOxideSensor_Unit4'
]
display(df[mo_cols].isna().sum())

In [ ]:
# Unit2 has ~14% missing values, impute with median
mo_median = df['MetalOxideSensor_Unit2'].median()
df['MetalOxideSensor_Unit2'] = df['MetalOxideSensor_Unit2'].fillna(mo_median)

In [ ]:
display(df[mo_cols].isna().sum())

### HVAC Operation Mode

In [ ]:
display(df['HVAC Operation Mode'].isna().sum())

### Ambient Light Level

In [ ]:
display(df['Ambient Light Level'].isna().sum())
display(df['Ambient Light Level'].value_counts())

### Activity Level

In [ ]:
display(df['Activity Level'].isna().sum())
display(df['Activity Level'].value_counts())